In [0]:
%run ./01_logging_utils.py

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

In [0]:

validation_id = new_batch_id("validation")
validation_start = datetime.now()

results = []

def check(name, condition, details):
    status = "PASS" if condition else "FAIL"
    results.append((name, status, str(details)))
    print(f"{status}: {name} | {details}")


try:
    # 1. Bronze and Silver must contain data
    bronze_count = spark.table(T_BRONZE).count()
    silver_count = spark.table(T_SILVER).count()

    check("Bronze is not empty", bronze_count > 0,
          f"rows={bronze_count}")

    check("Silver is not empty", silver_count > 0,
          f"rows={silver_count}")

    # 2. Silver should not contain duplicate coin/timestamp keys
    silver_duplicates = (
        spark.table(T_SILVER)
        .groupBy("coin_id", "event_ts")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    check("Silver duplicate keys", silver_duplicates == 0,
          f"duplicate_groups={silver_duplicates}")

    # 3. Silver should not contain invalid prices or missing keys
    invalid_silver = spark.table(T_SILVER).filter(
        F.col("coin_id").isNull()
        | F.col("event_ts").isNull()
        | F.col("price").isNull()
        | (F.col("price") < 0)
    ).count()

    check("Silver required fields and prices",
          invalid_silver == 0, f"invalid_rows={invalid_silver}")

    # 4. All expected Gold tables should contain data
    gold_tables = [
        T_DIM_COIN,
        T_DIM_DATE,
        T_DIM_HOUR,
        T_DIM_SCENARIO,
        T_BRIDGE,
        T_FACT_HOURLY,
        T_FACT_DAILY,
        T_FACT_PORTF
    ]

    for table_name in gold_tables:
        row_count = spark.table(table_name).count()
        check(
            f"Gold table populated: {table_name.split('.')[-1]}",
            row_count > 0,
            f"rows={row_count}"
        )

    # 5. Gold facts should not contain duplicate business keys
    duplicate_checks = [
        (T_FACT_HOURLY, ["coin_key", "event_ts"]),
        (T_FACT_DAILY, ["coin_key", "date_key"]),
        (T_FACT_PORTF, ["scenario_key", "date_key"])
    ]

    for table_name, keys in duplicate_checks:
        duplicates = (
            spark.table(table_name)
            .groupBy(*keys)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        check(
            f"Unique keys: {table_name.split('.')[-1]}",
            duplicates == 0,
            f"duplicate_groups={duplicates}"
        )

    # 6. Scenario weights should sum to approximately 1
    weight_results = (
        spark.table(T_BRIDGE)
        .groupBy("scenario_key")
        .agg(F.sum("weight").alias("weight_sum"))
        .collect()
    )

    bad_weights = [
        row["scenario_key"]
        for row in weight_results
        if row["weight_sum"] is None
        or abs(row["weight_sum"] - 1.0) > 1e-6
    ]

    check("Scenario weights sum to 1",
          len(bad_weights) == 0,
          f"scenarios_checked={len(weight_results)}, bad={len(bad_weights)}")

    # 7. Portfolio returns should not be null
    null_returns = spark.table(T_FACT_PORTF).filter(
        F.col("portfolio_return_pct").isNull()
    ).count()

    check("Portfolio returns populated",
          null_returns == 0,
          f"null_returns={null_returns}")

    # Save validation results
    result_df = spark.createDataFrame(
        results,
        ["check_name", "status", "details"]
    )

    display(result_df)

    passed = sum(1 for r in results if r[1] == "PASS")
    failed = sum(1 for r in results if r[1] == "FAIL")
    overall_status = "SUCCESS" if failed == 0 else "FAILED"

    log_run(
        validation_id,
        "validation",
        f"checks={len(results)} | passed={passed} | failed={failed}",
        validation_start,
        datetime.now(),
        overall_status,
        err=None if failed == 0 else f"{failed} validation checks failed"
    )

    print(f"\nValidation summary: {passed} passed, {failed} failed.")

    if failed > 0:
        raise Exception(
            f"Validation failed: {failed} of {len(results)} checks did not pass."
        )

except Exception as e:
    # Log unexpected errors but avoid logging a second failureif the validation summary has already been recorded.
    if not any(
        r[0] == "Portfolio returns populated"
        for r in results
    ):
        log_run(
            validation_id,
            "validation",
            "Validation execution error",
            validation_start,
            datetime.now(),
            "FAILED",
            err=str(e)[:500]
        )
    raise
